# Data Retention Management

This notebook implements automated data retention policies for the video telemetry QoS analytics pipeline.

## Retention Policies

* **Bronze Layer** (`bronze_telemetry_raw`): 90 days retention
* **Silver Layer** (`silver_telemetry_enriched`): 90 days retention
* **Silver Quarantine** (`silver_telemetry_quarantine`): 180 days retention
* **Gold Layer** (`gold_viewer_qos_metrics_streaming`): 365 days retention

## What This Notebook Does

1. **Configure Delta properties** - Set log and deleted file retention durations
2. **Vacuum tables** - Remove old data files beyond retention periods
3. **Delete quarantine records** - Clean up old quarantine data
4. **Optimize tables** - Compact small files for better performance
5. **Verify retention status** - Check current data ranges and storage metrics

## Usage

Run all cells in sequence. Recommended to schedule as a weekly job to maintain optimal storage usage.

In [0]:
# Configuration for data retention management
import datetime
from pyspark.sql import functions as F

# Catalog and schema
CATALOG = "workspace"
SCHEMA = "hive_video_analytics"

# Table names
TABLES = {
    "bronze": f"{CATALOG}.{SCHEMA}.bronze_telemetry_raw",
    "silver": f"{CATALOG}.{SCHEMA}.silver_telemetry_enriched",
    "quarantine": f"{CATALOG}.{SCHEMA}.silver_telemetry_quarantine",
    "gold": f"{CATALOG}.{SCHEMA}.gold_viewer_qos_metrics_streaming"
}

# Retention periods (in days)
RETENTION_PERIODS = {
    "bronze": 90,
    "silver": 90,
    "quarantine": 180,
    "gold": 365
}

# Delta log retention (in hours) - keep longer than data retention for time travel
LOG_RETENTION_HOURS = {
    "bronze": 90 * 24,      # 90 days
    "silver": 90 * 24,      # 90 days
    "quarantine": 180 * 24, # 180 days
    "gold": 365 * 24        # 365 days
}

print("Configuration loaded successfully")
print(f"Catalog: {CATALOG}")
print(f"Schema: {SCHEMA}")
print(f"Tables: {list(TABLES.keys())}")
print(f"Retention periods: {RETENTION_PERIODS}")

In [0]:
%sql
-- Set Delta log retention and deleted file retention durations for all tables
-- These properties control how long Delta keeps transaction logs and tombstone files

-- Bronze table: 90 days retention
ALTER TABLE workspace.hive_video_analytics.bronze_telemetry_raw 
SET TBLPROPERTIES (
  'delta.logRetentionDuration' = '2160 hours',
  'delta.deletedFileRetentionDuration' = '2160 hours'
);

-- Silver table: 90 days retention
ALTER TABLE workspace.hive_video_analytics.silver_telemetry_enriched 
SET TBLPROPERTIES (
  'delta.logRetentionDuration' = '2160 hours',
  'delta.deletedFileRetentionDuration' = '2160 hours'
);

-- Quarantine table: 180 days retention
ALTER TABLE workspace.hive_video_analytics.silver_telemetry_quarantine 
SET TBLPROPERTIES (
  'delta.logRetentionDuration' = '4320 hours',
  'delta.deletedFileRetentionDuration' = '4320 hours'
);

-- Gold table: 365 days retention
ALTER TABLE workspace.hive_video_analytics.gold_viewer_qos_metrics_streaming 
SET TBLPROPERTIES (
  'delta.logRetentionDuration' = '8760 hours',
  'delta.deletedFileRetentionDuration' = '8760 hours'
);

In [0]:
# VACUUM bronze_telemetry_raw - Remove old data files
try:
    print(f"Starting VACUUM operation on {TABLES['bronze']}...")
    print(f"Retention period: {RETENTION_PERIODS['bronze']} days")
    
    retention_hours = RETENTION_PERIODS['bronze'] * 24
    
    spark.sql(f"VACUUM {TABLES['bronze']} RETAIN {retention_hours} HOURS")
    
    print(f"✓ Successfully vacuumed {TABLES['bronze']}")
    print(f"  Removed files older than {RETENTION_PERIODS['bronze']} days")
    
except Exception as e:
    print(f"✗ Error vacuuming {TABLES['bronze']}: {str(e)}")
    raise

In [0]:
# VACUUM silver_telemetry_enriched - Remove old data files
try:
    print(f"Starting VACUUM operation on {TABLES['silver']}...")
    print(f"Retention period: {RETENTION_PERIODS['silver']} days")
    
    retention_hours = RETENTION_PERIODS['silver'] * 24
    
    spark.sql(f"VACUUM {TABLES['silver']} RETAIN {retention_hours} HOURS")
    
    print(f"✓ Successfully vacuumed {TABLES['silver']}")
    print(f"  Removed files older than {RETENTION_PERIODS['silver']} days")
    
except Exception as e:
    print(f"✗ Error vacuuming {TABLES['silver']}: {str(e)}")
    raise

In [0]:
# VACUUM silver_telemetry_quarantine - Remove old data files
try:
    print(f"Starting VACUUM operation on {TABLES['quarantine']}...")
    print(f"Retention period: {RETENTION_PERIODS['quarantine']} days")
    
    retention_hours = RETENTION_PERIODS['quarantine'] * 24
    
    spark.sql(f"VACUUM {TABLES['quarantine']} RETAIN {retention_hours} HOURS")
    
    print(f"✓ Successfully vacuumed {TABLES['quarantine']}")
    print(f"  Removed files older than {RETENTION_PERIODS['quarantine']} days")
    
except Exception as e:
    print(f"✗ Error vacuuming {TABLES['quarantine']}: {str(e)}")
    raise

In [0]:
%sql
-- Delete quarantine records older than 180 days
-- This removes actual data records, not just files

DELETE FROM workspace.hive_video_analytics.silver_telemetry_quarantine
WHERE timestamp < current_timestamp() - INTERVAL 180 DAYS;

In [0]:
%sql
-- OPTIMIZE tables after cleanup to compact small files
-- This improves query performance and reduces storage overhead

OPTIMIZE workspace.hive_video_analytics.bronze_telemetry_raw;
OPTIMIZE workspace.hive_video_analytics.silver_telemetry_enriched;
OPTIMIZE workspace.hive_video_analytics.silver_telemetry_quarantine;
OPTIMIZE workspace.hive_video_analytics.gold_viewer_qos_metrics_streaming;

In [0]:
%sql
-- Verify Delta table properties for all tables
-- Check that retention durations are correctly set

SELECT 
  'bronze_telemetry_raw' as table_name,
  key,
  value
FROM (SHOW TBLPROPERTIES workspace.hive_video_analytics.bronze_telemetry_raw)
WHERE key LIKE 'delta.%Retention%'

UNION ALL

SELECT 
  'silver_telemetry_enriched' as table_name,
  key,
  value
FROM (SHOW TBLPROPERTIES workspace.hive_video_analytics.silver_telemetry_enriched)
WHERE key LIKE 'delta.%Retention%'

UNION ALL

SELECT 
  'silver_telemetry_quarantine' as table_name,
  key,
  value
FROM (SHOW TBLPROPERTIES workspace.hive_video_analytics.silver_telemetry_quarantine)
WHERE key LIKE 'delta.%Retention%'

UNION ALL

SELECT 
  'gold_viewer_qos_metrics_streaming' as table_name,
  key,
  value
FROM (SHOW TBLPROPERTIES workspace.hive_video_analytics.gold_viewer_qos_metrics_streaming)
WHERE key LIKE 'delta.%Retention%'

ORDER BY table_name, key;

In [0]:
%sql
-- Check current data retention status for all tables
-- Shows oldest and newest records, plus total counts

SELECT 
  'bronze_telemetry_raw' as table_name,
  MIN(timestamp) as oldest_record,
  MAX(timestamp) as newest_record,
  DATEDIFF(MAX(timestamp), MIN(timestamp)) as data_span_days,
  COUNT(*) as record_count
FROM workspace.hive_video_analytics.bronze_telemetry_raw

UNION ALL

SELECT 
  'silver_telemetry_enriched' as table_name,
  MIN(timestamp) as oldest_record,
  MAX(timestamp) as newest_record,
  DATEDIFF(MAX(timestamp), MIN(timestamp)) as data_span_days,
  COUNT(*) as record_count
FROM workspace.hive_video_analytics.silver_telemetry_enriched

UNION ALL

SELECT 
  'silver_telemetry_quarantine' as table_name,
  MIN(timestamp) as oldest_record,
  MAX(timestamp) as newest_record,
  DATEDIFF(MAX(timestamp), MIN(timestamp)) as data_span_days,
  COUNT(*) as record_count
FROM workspace.hive_video_analytics.silver_telemetry_quarantine

UNION ALL

SELECT 
  'gold_viewer_qos_metrics_streaming' as table_name,
  MIN(window_start) as oldest_record,
  MAX(window_start) as newest_record,
  DATEDIFF(MAX(window_start), MIN(window_start)) as data_span_days,
  COUNT(*) as record_count
FROM workspace.hive_video_analytics.gold_viewer_qos_metrics_streaming

ORDER BY table_name;

In [0]:
%sql
-- Get detailed storage metrics for all tables
-- Shows table size, number of files, and format

DESCRIBE DETAIL workspace.hive_video_analytics.bronze_telemetry_raw
UNION ALL
DESCRIBE DETAIL workspace.hive_video_analytics.silver_telemetry_enriched
UNION ALL
DESCRIBE DETAIL workspace.hive_video_analytics.silver_telemetry_quarantine
UNION ALL
DESCRIBE DETAIL workspace.hive_video_analytics.gold_viewer_qos_metrics_streaming;

# Summary and Scheduling

## What Was Done

This notebook has successfully:

1. ✓ Configured Delta table properties for log and file retention
2. ✓ Vacuumed old data files from all tables based on retention policies
3. ✓ Deleted old quarantine records beyond 180 days
4. ✓ Optimized all tables for better performance
5. ✓ Verified retention settings and data status

## Storage Savings

Check the storage metrics above to see the reduction in file count and storage size after cleanup.

## Scheduling This Notebook

To maintain optimal storage usage, schedule this notebook to run weekly:

### Option 1: Using Databricks Jobs UI

1. Go to **Workflows** → **Jobs**
2. Click **Create Job**
3. Add this notebook as a task
4. Set schedule: **Weekly, every Sunday at 2:00 AM**
5. Configure email alerts for failures
6. Save and enable the job

### Option 2: Using Databricks CLI

```bash
databricks jobs create --json '{
  "name": "Data Retention Management - Weekly",
  "tasks": [{
    "task_key": "retention_cleanup",
    "notebook_task": {
      "notebook_path": "/Users/vasuambi@gmail.com/video_telemetry_qos_analytics_pipeline_1ff1d722/operations/Data_Retention_Management"
    },
    "existing_cluster_id": "your-cluster-id"
  }],
  "schedule": {
    "quartz_cron_expression": "0 0 2 ? * SUN",
    "timezone_id": "UTC"
  }
}'
```

## Monitoring

* Monitor job run history for failures
* Review storage metrics trends over time
* Adjust retention periods if business requirements change
* Alert on unexpected data age or storage growth